In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import torch, esm, pandas as pd, numpy as np
from tqdm.auto import tqdm

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

print("Loading ESM-2 650M (cached from W5) ...")
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval().to(device)

seqs = pd.read_csv("../data/sequences.csv")
seq_by_acc = dict(zip(seqs['acc'], seqs['sequence']))

regions = pd.read_csv("../data/regions_master.csv")
print(f"Regions to embed: {len(regions)}")

MAX_LEN = 1022

@torch.no_grad()
def per_residue_embeddings(seq):
    s = seq[:MAX_LEN]
    _, _, toks = batch_converter([("p", s)])
    toks = toks.to(device)
    out = model(toks, repr_layers=[33])
    return out["representations"][33][0, 1:-1]  # (L, 1280), strips CLS + EOS

# Loop per protein, then pool per region within that protein
region_embeddings = {}
fallbacks = []
for acc in tqdm(regions['acc'].unique(), total=regions['acc'].nunique()):
    seq = seq_by_acc.get(acc)
    if seq is None:
        continue
    per_res = per_residue_embeddings(seq)  # (L, 1280) on device
    L = per_res.shape[0]
    for _, r in regions[regions['acc'] == acc].iterrows():
        start, end = r['start'], r['end']
        if start > L:  # entire region beyond truncation window
            emb = per_res.mean(dim=0).cpu().numpy().astype("float32")
            fallbacks.append(r['region_id'])
        else:
            e = min(end, L)
            emb = per_res[start-1:e].mean(dim=0).cpu().numpy().astype("float32")
        region_embeddings[r['region_id']] = emb
    del per_res  # release GPU memory before next protein

region_ids = list(region_embeddings.keys())
X = np.stack([region_embeddings[rid] for rid in region_ids])
np.savez_compressed("../data/features_esm2_region.npz",
                    region_ids=np.array(region_ids), X=X.astype("float32"))
print(f"\nSaved {X.shape[0]} region embeddings of dim {X.shape[1]}")
print(f"Truncation fallbacks (region beyond MAX_LEN): {len(fallbacks)}")

Using device: mps
Loading ESM-2 650M (cached from W5) ...
Regions to embed: 3231


  0%|          | 0/1278 [00:00<?, ?it/s]


Saved 3231 region embeddings of dim 1280
Truncation fallbacks (region beyond MAX_LEN): 178


In [2]:
# After the loop finishes:
missing = set(regions['region_id']) - set(region_embeddings.keys())
print(f"Regions missing from embeddings: {len(missing)}")
if missing:
    print(f"  Sample missing IDs: {list(missing)[:5]}")

Regions missing from embeddings: 0


In [3]:
data = np.load("../data/features_esm2_region.npz", allow_pickle=True)
X, rids = data['X'], data['region_ids']
norms = np.linalg.norm(X, axis=1)
print("shape:", X.shape, "dtype:", X.dtype)
print(f"L2 norms: min={norms.min():.2f}, max={norms.max():.2f}, mean={norms.mean():.2f}")
print(f"NaNs? {np.isnan(X).any()}; zero-norm? {(norms==0).sum()}")

shape: (3231, 1280) dtype: float32
L2 norms: min=4.77, max=10.24, mean=8.82
NaNs? False; zero-norm? 0


In [5]:
single = state[state['n_regions_in_protein'] == 1].copy()
print(f"Single-region proteins: {len(single)}, "
      f"positives: {single['d2o_label'].sum()}, "
      f"positive rate: {single['d2o_label'].mean():.3f}")
single.to_csv("../data/regions_single_region.csv", index=False)

NameError: name 'state' is not defined

In [6]:
state = pd.read_csv("../data/regions_master.csv")

In [7]:
single = state[state['n_regions_in_protein'] == 1].copy()
print(f"Single-region proteins: {len(single)}, "
      f"positives: {single['d2o_label'].sum()}, "
      f"positive rate: {single['d2o_label'].mean():.3f}")
single.to_csv("../data/regions_single_region.csv", index=False)

Single-region proteins: 530, positives: 39, positive rate: 0.074


In [8]:
# Control 1 (single-region-protein subset) numbers logged for later:
#   n = 530 regions, positives = 39 (7.4% rate)
# Will run the factorial on this subset ONLY IF the full W9.4 factorial 
# produces a significant positive result — needed to rule out the 
# GO-fingerprint-per-protein confound. Otherwise Control 1 is 
# underpowered and unnecessary.